# 03. Circuit Resource Analysis and Transform Validation

This notebook separates two claims: (1) the Qiskit QFT convention matches the NumPy FFT split step used for the harmonic oscillator, and (2) resource counts are small-system exact-synthesis or analytical estimates, not asymptotic optimal decompositions. The infinite-well resource model is reported for a quantum-sine-transform implementation rather than a periodic QFT circuit.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
COMMON_NOTEBOOK = str(PROJECT_ROOT / "notebooks" / "00_common_qft_split_functions.ipynb")
%run "$COMMON_NOTEBOOK"

FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
configure_matplotlib()

In [2]:
N = 64
hbar = 1.0
m = 1.0
omega = 1.0

harmonic_domain = (-8.0, 8.0)
harmonic_t_max = 2.0 * np.pi
harmonic_r_default = 100

well_L = 10.0
well_t_max = 6.0
well_r_default = 100

basis_gates = ["rx", "ry", "rz", "cx"]

In [3]:
validation_records = []

x_h, dx_h = periodic_grid(harmonic_domain[0], harmonic_domain[1], N)
p_h = fft_momentum_grid(N, dx=dx_h, hbar=hbar)
dt_h = harmonic_t_max / harmonic_r_default
position_phase_h = np.exp(-0.5j * harmonic_potential(x_h, mass=m, omega=omega) * dt_h / hbar)
momentum_phase_h = np.exp(-1j * (p_h**2 / (2.0 * m)) * dt_h / hbar)
validation_records.append(qft_split_circuit_validation(position_phase_h, momentum_phase_h))

dt_w = well_t_max / well_r_default
validation_records.append(sine_transform_validation(N, length=well_L, mass=m, hbar=hbar, dt=dt_w))

validation_df = pd.DataFrame(validation_records)
save_dataframe(validation_df, TABLES_DIR, "circuit_validation.csv")
display(Markdown("## Transform Validation"))
display(validation_df)

if not validation_df["passed"].all():
    raise RuntimeError("At least one transform validation failed.")

## Transform Validation

,validation,n_qubits,direct_l2_error,phase_adjusted_l2_error,passed,grid_size
0,qiskit_qft_matches_numpy_fft_step,6.0,9.903349e-15,7.506341e-15,True,NaN
1,dst_matrix_matches_scipy_sine_step,NaN,4.148257e-16,NaN,True,64.0


In [4]:
harmonic_logical = build_periodic_resource_circuit(position_phase_h, momentum_phase_h)
harmonic_transpiled, harmonic_metrics, harmonic_breakdown = transpile_and_extract_metrics(harmonic_logical, basis_gates)

infinite_well_logical = build_infinite_well_circuit(grid_size=N, length=well_L, mass=m, hbar=hbar, dt=dt_w)
infinite_well_transpiled, well_metrics, well_breakdown = transpile_and_extract_metrics(infinite_well_logical, basis_gates)

n_qubits = int(np.log2(N))

single_step_metrics_df = pd.DataFrame(
    [
        {
            "system": "harmonic_oscillator",
            "N": N,
            "n_qubits": n_qubits,
            "dt": dt_h,
            **harmonic_metrics,
            "transform": "periodic_QFT",
            "synthesis_model": "Qiskit_transpiled_DiagonalGate_plus_QFTGate",
        },
        {
            "system": "infinite_well",
            "N": N,
            "n_qubits": n_qubits,
            "dt": dt_w,
            **well_metrics,
            "transform": "Dirichlet_quantum_sine_transform_QFT_extension",
            "synthesis_model": "Qiskit_transpiled_QST_proxy_with_QFT_extension",
        },
    ]
)

gate_breakdown_records = [
    {"system": "harmonic_oscillator", "gate_name": gate_name, "count": gate_count}
    for gate_name, gate_count in sorted(harmonic_breakdown.items())
]
gate_breakdown_records.extend(
    {"system": "infinite_well", "gate_name": gate_name, "count": gate_count}
    for gate_name, gate_count in sorted(well_breakdown.items())
)
gate_breakdown_df = pd.DataFrame(gate_breakdown_records)

save_dataframe(single_step_metrics_df, TABLES_DIR, "circuit_single_step_metrics.csv")
save_dataframe(gate_breakdown_df, TABLES_DIR, "circuit_gate_breakdown.csv")

display(Markdown("## Single-Step Metrics"))
display(single_step_metrics_df)
display(Markdown("## Gate Breakdown"))
display(gate_breakdown_df)

## Single-Step Metrics

,system,N,n_qubits,dt,single_step_1q_count,single_step_2q_count,single_step_depth,transform,synthesis_model
0,harmonic_oscillator,64,6,0.062832,213,264,354,periodic_QFT,Qiskit_transpiled_DiagonalGate_plus_QFTGate
1,infinite_well,64,6,0.060000,214,198,193,Dirichlet_quantum_sine_transform_QFT_extension,Qiskit_transpiled_QST_proxy_with_QFT_extension


## Gate Breakdown

,system,gate_name,count
0,harmonic_oscillator,barrier,4
1,harmonic_oscillator,cx,264
2,harmonic_oscillator,rx,6
3,harmonic_oscillator,ry,12
4,harmonic_oscillator,rz,195
5,infinite_well,barrier,2
6,infinite_well,cx,198
7,infinite_well,rx,8
8,infinite_well,ry,16
9,infinite_well,rz,190


In [5]:
draw_and_save_circuit(harmonic_logical, FIGURES_DIR, "harmonic_single_step_logical_circuit", scale=0.75, fold=40)
draw_and_save_circuit(harmonic_transpiled, FIGURES_DIR, "harmonic_single_step_transpiled_circuit", scale=0.55, fold=50)

for stale_path in FIGURES_DIR.glob("infinite_well_single_step_*_qiskit.*"):
    stale_path.unlink()
draw_and_save_circuit(infinite_well_logical, FIGURES_DIR, "infinite_well_single_step_logical_circuit", scale=0.75, fold=40)
draw_and_save_circuit(infinite_well_transpiled, FIGURES_DIR, "infinite_well_single_step_transpiled_circuit", scale=0.55, fold=50)



def save_block_diagram(stem: str, title: str, blocks: list[str]) -> None:
    fig, axis = plt.subplots(figsize=(10.0, 2.0), constrained_layout=True)
    axis.set_axis_off()

    box_style = {
        "boxstyle": "round,pad=0.6",
        "facecolor": "#ebf5fb", 
        "edgecolor": "#2874a6", 
        "linewidth": 2.0,
    }

    if len(blocks) == 3:
        x_positions = np.linspace(0.25, 0.75, len(blocks))
    else:
        x_positions = np.linspace(0.12, 0.88, len(blocks))

    for x_pos, label in zip(x_positions, blocks):
        axis.text(
            x_pos, 0.5, label,
            ha="center", va="center",
            fontsize=13, fontweight="bold", color="#1b4f72",
            bbox=box_style,
        )

    for i in range(len(blocks) - 1):
        left_text = blocks[i]
        right_text = blocks[i+1]

        left_hw = max(0.04, len(left_text) * 0.005 + 0.03)
        right_hw = max(0.04, len(right_text) * 0.005 + 0.03)

        axis.annotate(
            "", 
            xy=(x_positions[i+1] - right_hw, 0.5), 
            xytext=(x_positions[i] + left_hw, 0.5), 
            arrowprops={
                "arrowstyle": "-|>,head_length=0.8,head_width=0.4", 
                "lw": 2.5, 
                "color": "#2874a6"
            }
        )

    axis.set_title(title, fontsize=15, fontweight="bold", pad=15)
    save_publication_figure(fig, FIGURES_DIR, stem)
    plt.close(fig)

print("Saved compact and transpiled circuit diagrams.")

Saved compact and transpiled circuit diagrams.


In [6]:
fidelity_tables = {
    "harmonic_oscillator": TABLES_DIR / "harmonic_fidelity_vs_r.csv",
    "infinite_well": TABLES_DIR / "infinite_well_fidelity_vs_r.csv",
}

resource_totals_frames = []
for system_key, fidelity_path in fidelity_tables.items():
    if not fidelity_path.exists():
        raise FileNotFoundError(f"Missing {fidelity_path.name}. Run the corresponding dynamics notebook first.")
    fidelity_df = pd.read_csv(fidelity_path)
    metrics_row = single_step_metrics_df.loc[single_step_metrics_df["system"] == system_key].iloc[0]
    totals_df = fidelity_df.copy()
    totals_df["single_step_1q_count"] = int(metrics_row["single_step_1q_count"])
    totals_df["single_step_2q_count"] = int(metrics_row["single_step_2q_count"])
    totals_df["single_step_depth"] = int(metrics_row["single_step_depth"])
    totals_df["synthesis_model"] = metrics_row["synthesis_model"]
    totals_df["total_1q_count"] = totals_df["single_step_1q_count"] * totals_df["r"]
    totals_df["total_2q_count"] = totals_df["single_step_2q_count"] * totals_df["r"]
    totals_df["total_gate_count"] = totals_df["total_1q_count"] + totals_df["total_2q_count"]
    totals_df["estimated_total_depth"] = totals_df["single_step_depth"] * totals_df["r"]
    resource_totals_frames.append(totals_df)
    save_dataframe(totals_df, TABLES_DIR, f"{system_key}_resource_totals_vs_r.csv")

resource_totals_df = pd.concat(resource_totals_frames, ignore_index=True)
save_dataframe(resource_totals_df, TABLES_DIR, "resource_totals_vs_r.csv")
display(Markdown("## Resource Totals Across Fidelity Sweeps"))
display(resource_totals_df)

## Resource Totals Across Fidelity Sweeps

,system,r,dt,final_time_fidelity,minimum_fidelity,mean_fidelity,single_step_1q_count,single_step_2q_count,single_step_depth,synthesis_model,total_1q_count,total_2q_count,total_gate_count,estimated_total_depth
0,harmonic_oscillator,20,0.314159,0.997158,0.996259,0.998741,213,264,354,Qiskit_transpiled_DiagonalGate_plus_QFTGate,4260,5280,9540,7080
1,harmonic_oscillator,40,0.157080,0.999822,0.999767,0.999922,213,264,354,Qiskit_transpiled_DiagonalGate_plus_QFTGate,8520,10560,19080,14160
2,harmonic_oscillator,60,0.104720,0.999965,0.999954,0.999985,213,264,354,Qiskit_transpiled_DiagonalGate_plus_QFTGate,12780,15840,28620,21240
3,harmonic_oscillator,80,0.078540,0.999989,0.999985,0.999995,213,264,354,Qiskit_transpiled_DiagonalGate_plus_QFTGate,17040,21120,38160,28320
4,harmonic_oscillator,100,0.062832,0.999995,0.999994,0.999998,213,264,354,Qiskit_transpiled_DiagonalGate_plus_QFTGate,21300,26400,47700,35400
5,infinite_well,20,0.300000,1.000000,1.000000,1.000000,214,198,193,Qiskit_transpiled_QST_proxy_with_QFT_extension,4280,3960,8240,3860
6,infinite_well,40,0.150000,1.000000,1.000000,1.000000,214,198,193,Qiskit_transpiled_QST_proxy_with_QFT_extension,8560,7920,16480,7720
7,infinite_well,60,0.100000,1.000000,1.000000,1.000000,214,198,193,Qiskit_transpiled_QST_proxy_with_QFT_extension,12840,11880,24720,11580
8,infinite_well,80,0.075000,1.000000,1.000000,1.000000,214,198,193,Qiskit_transpiled_QST_proxy_with_QFT_extension,17120,15840,32960,15440
9,infinite_well,100,0.060000,1.000000,1.000000,1.000000,214,198,193,Qiskit_transpiled_QST_proxy_with_QFT_extension,21400,19800,41200,19300


In [7]:
harmonic_resource_df = resource_totals_df.loc[resource_totals_df["system"] == "harmonic_oscillator"].reset_index(drop=True)
well_resource_df = resource_totals_df.loc[resource_totals_df["system"] == "infinite_well"].reset_index(drop=True)

figures = [
    (plot_gate_counts(harmonic_resource_df, "Harmonic oscillator gate counts"), "harmonic_gate_counts_vs_steps"),
    (plot_gate_counts(well_resource_df, "Infinite-well gate counts"), "infinite_well_gate_counts_vs_steps"),
    (plot_fidelity_vs_gate_count(harmonic_resource_df, "total_2q_count", "Total CX count", "Harmonic fidelity vs CX count"), "harmonic_fidelity_vs_two_qubit_gate_count"),
    (plot_fidelity_vs_gate_count(harmonic_resource_df, "total_gate_count", "Total 1q + 2q gate count", "Harmonic fidelity vs total gate count"), "harmonic_fidelity_vs_total_gate_count"),
    (plot_fidelity_vs_gate_count(well_resource_df, "total_2q_count", "Total CX count", "Infinite-well fidelity vs CX count"), "infinite_well_fidelity_vs_two_qubit_gate_count"),
    (plot_fidelity_vs_gate_count(well_resource_df, "total_gate_count", "Total 1q + 2q gate count", "Infinite-well fidelity vs total gate count"), "infinite_well_fidelity_vs_total_gate_count"),
]
for figure, stem in figures:
    save_publication_figure(figure, FIGURES_DIR, stem)
    plt.close(figure)

print("Saved gate-count and fidelity-vs-gate-count figures.")

Saved gate-count and fidelity-vs-gate-count figures.
